In [1]:
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats

base = Path('/workspace')
df = pd.read_csv(base/'t3-prism-bo-batch-drop-results.csv')
print('shape:', df.shape)
print(df[['specimen','n_valid','t180_mean','t180_sd','e_rebound_mean','fn_hz_mean','mass_g','spec']].to_string(index=False))
print('\nRelevant figure/source files:')
for p in sorted(base.rglob('*')):
    if any(k in p.name.lower() for k in ['pareto','loocv','feature-importance']) and p.is_file():
        print(p, p.stat().st_size)


shape: (8, 28)
specimen  n_valid  t180_mean  t180_sd  e_rebound_mean  fn_hz_mean  mass_g spec
  6lhxfy      101   0.893078 0.004155        0.050376  368.376693   18.50   01
  6nheas      101   0.997008 0.003208        0.040246  321.991958   21.73   05
  9hhbkp      101   1.018336 0.001725        0.021501  364.088245   21.62   00
  amdjwm      101   0.980495 0.002509        0.029618  455.142284     NaN  NaN
  autv5r      103   1.040430 0.003486        0.026810  385.715265   22.04   02
  bag26v      101   1.061620 0.005120        0.024098         NaN   21.42   08
  bpx68c      101   1.011072 0.002360        0.020442  467.572299   20.23   S0
  nvxsrv      101   1.027549 0.004379        0.026564  294.253070   20.66   04

Relevant figure/source files:
/workspace/t3-prism-bo-round1-pareto.png 303519


In [2]:
import pandas as pd, numpy as np
from scipy import stats
from pathlib import Path

base=Path('/workspace')
df=pd.read_csv(base/'t3-prism-bo-batch-drop-results.csv')
h_m=60*0.0254
g=9.80665
df['E_calc_mJ']=df['e_rebound_mean']*df['mass_g']*g*h_m
cols=['specimen','mass_g','e_rebound_mean','E_calc_mJ','t180_mean','t180_sd','fn_hz_mean']
print(df[cols].sort_values('t180_mean').to_string(index=False, float_format=lambda x:f'{x:.8f}'))

valid=df.dropna(subset=['mass_g']).copy()
r,p=stats.pearsonr(valid.mass_g, valid.t180_mean)
print(f'\nMass correlation articles: {valid.specimen.tolist()}')
print(f'Pearson r={r:.12f}, p(two-sided)={p:.12f}, n={len(valid)}')
mn,mx=df.t180_mean.min(),df.t180_mean.max()
print(f'Range={mx-mn:.12f}; /min={(mx-mn)/mn*100:.6f}%; /max={(mx-mn)/mx*100:.6f}%; /mean={(mx-mn)/df.t180_mean.mean()*100:.6f}%')
print('Above 1:', df.loc[df.t180_mean>1,'specimen'].tolist(), 'count', (df.t180_mean>1).sum())
print('Below 1:', df.loc[df.t180_mean<1,'specimen'].tolist(), 'count', (df.t180_mean<1).sum())

# exact nondominated set for minimization among rows with both objectives
pts=valid[['specimen','t180_mean','E_calc_mJ']].copy()
nd=[]; dominators={}
for i,row in pts.iterrows():
    dom=pts[((pts.t180_mean <= row.t180_mean)&(pts.E_calc_mJ <= row.E_calc_mJ)) &
            ((pts.t180_mean < row.t180_mean)|(pts.E_calc_mJ < row.E_calc_mJ))]
    dominators[row.specimen]=dom.specimen.tolist()
    if dom.empty: nd.append(row.specimen)
print('\nNon-dominated:', nd)
for k,v in dominators.items(): print(k, '<- dominated by',v)


specimen      mass_g  e_rebound_mean   E_calc_mJ  t180_mean    t180_sd   fn_hz_mean
  6lhxfy 18.50000000      0.05037622 13.92845551 0.89307779 0.00415544 368.37669298
  amdjwm         NaN      0.02961781         NaN 0.98049532 0.00250869 455.14228427
  6nheas 21.73000000      0.04024595 13.07036008 0.99700829 0.00320752 321.99195772
  bpx68c 20.23000000      0.02044175  6.18044319 1.01107166 0.00236013 467.57229922
  9hhbkp 21.62000000      0.02150119  6.94742490 1.01833601 0.00172505 364.08824528
  nvxsrv 20.66000000      0.02656356  8.20204807 1.02754920 0.00437925 294.25307034
  autv5r 22.04000000      0.02680957  8.83094236 1.04043034 0.00348606 385.71526459
  bag26v 21.42000000      0.02409813  7.71451090 1.06161977 0.00512009          NaN

Mass correlation articles: ['6lhxfy', '6nheas', '9hhbkp', 'autv5r', 'bag26v', 'bpx68c', 'nvxsrv']
Pearson r=0.829126380869, p(two-sided)=0.021095146052, n=7
Range=0.168541986041; /min=18.872039%; /max=15.875928%; /mean=16.792092%
Above 1: ['9h

In [3]:
# Produce exact manuscript-table vs CSV rounding checks and candidate relative-spread definitions.
import pandas as pd, numpy as np
from pathlib import Path
from scipy import stats

df = pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
g=9.80665; h=60*0.0254
df['E_mJ']=df.e_rebound_mean*df.mass_g*g*h
# Values transcribed from Table 5
T = pd.DataFrame([
('6lhxfy',18.50,.8931,.0042,13.9,368),('6nheas',21.73,.9970,.0032,13.1,322),
('amdjwm',np.nan,.9805,.0025,np.nan,455),('bpx68c',20.23,1.0111,.0024,6.2,468),
('9hhbkp',21.62,1.0183,.0017,7.0,364),('nvxsrv',20.66,1.0275,.0044,8.2,294),
('autv5r',22.04,1.0404,.0035,8.8,386),('bag26v',21.42,1.0616,.0051,7.7,np.nan)],
columns=['specimen','mass_tab','mean_tab','sd_tab','E_tab','fn_tab'])
z=T.merge(df[['specimen','mass_g','t180_mean','t180_sd','E_mJ','fn_hz_mean','n_valid']],on='specimen')
z['mean_ok']=z.mean_tab.eq(z.t180_mean.round(4)); z['sd_ok']=z.sd_tab.eq(z.t180_sd.round(4)); z['E_ok']=(z.E_tab.eq(z.E_mJ.round(1))|z.E_tab.isna()&z.E_mJ.isna()); z['fn_ok']=(z.fn_tab.eq(z.fn_hz_mean.round(0))|z.fn_tab.isna()&z.fn_hz_mean.isna())
print(z[['specimen','n_valid','E_mJ','E_tab','mean_tab','t180_mean','sd_tab','t180_sd','fn_tab','fn_hz_mean','mean_ok','sd_ok','E_ok','fn_ok']].to_string(index=False))
print('\nAll displayed values match prescribed rounding:', z[['mean_ok','sd_ok','E_ok','fn_ok']].all().all())
# suggestion non-dominated set among orange points
s=pd.read_csv('/workspace/t3-prism-bo-suggestions-round1.csv')
nd=[]
for _,r in s.iterrows():
 d=s[((s.pred_t180_mean<=r.pred_t180_mean)&(s.pred_e_reb_mJ_mean<=r.pred_e_reb_mJ_mean))&((s.pred_t180_mean<r.pred_t180_mean)|(s.pred_e_reb_mJ_mean<r.pred_e_reb_mJ_mean))]
 if d.empty: nd.append(int(r.trial_index))
print('Non-dominated suggested trials:',nd)
print('Suggested coordinate ranges:',s.pred_t180_mean.min(),s.pred_t180_mean.max(),s.pred_e_reb_mJ_mean.min(),s.pred_e_reb_mJ_mean.max())


specimen  n_valid      E_mJ  E_tab  mean_tab  t180_mean  sd_tab  t180_sd  fn_tab  fn_hz_mean  mean_ok  sd_ok  E_ok  fn_ok
  6lhxfy      101 13.928456   13.9    0.8931   0.893078  0.0042 0.004155   368.0  368.376693     True   True  True   True
  6nheas      101 13.070360   13.1    0.9970   0.997008  0.0032 0.003208   322.0  321.991958     True   True  True   True
  amdjwm      101       NaN    NaN    0.9805   0.980495  0.0025 0.002509   455.0  455.142284     True   True  True   True
  bpx68c      101  6.180443    6.2    1.0111   1.011072  0.0024 0.002360   468.0  467.572299     True   True  True   True
  9hhbkp      101  6.947425    7.0    1.0183   1.018336  0.0017 0.001725   364.0  364.088245     True   True False   True
  nvxsrv      101  8.202048    8.2    1.0275   1.027549  0.0044 0.004379   294.0  294.253070     True   True  True   True
  autv5r      103  8.830942    8.8    1.0404   1.040430  0.0035 0.003486   386.0  385.715265     True   True  True   True
  bag26v      101  7.714

In [4]:
import pandas as pd, numpy as np
from scipy import stats

df=pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
valid=df.dropna(subset=['mass_g'])
r,p=stats.pearsonr(valid.mass_g,valid.t180_mean)
# Fisher z CI
z=np.arctanh(r); se=1/np.sqrt(len(valid)-3)
ci=np.tanh(z+np.array([-1,1])*stats.norm.ppf(.975)*se)
df['cv_pct']=100*df.t180_sd/df.t180_mean
print(f'Pearson r={r:.6f}, p={p:.6f}, 95% Fisher CI=({ci[0]:.3f}, {ci[1]:.3f})')
print('Within-article CVs (%):')
print(df[['specimen','cv_pct']].sort_values('cv_pct').to_string(index=False,float_format=lambda x:f'{x:.3f}'))
# Check alternative gravity conventions for disputed 9hhbkp energy
row=df[df.specimen=='9hhbkp'].iloc[0]
for gg in [9.80665,9.81]:
 print(gg,row.e_rebound_mean*row.mass_g*gg*(60*0.0254))


Pearson r=0.829126, p=0.021095, 95% Fisher CI=(0.203, 0.974)
Within-article CVs (%):
specimen  cv_pct
  9hhbkp   0.169
  bpx68c   0.233
  amdjwm   0.256
  6nheas   0.322
  autv5r   0.335
  nvxsrv   0.426
  6lhxfy   0.465
  bag26v   0.482
9.80665 6.947424898322795
9.81 6.94979817292823


In [5]:
from scipy import stats
import numpy as np
for r in [0.70,-0.12]:
    n=7; dfree=n-2
    t=r*np.sqrt(dfree/(1-r*r))
    p=2*stats.t.sf(abs(t),dfree)
    z=np.arctanh(r); se=1/np.sqrt(n-3)
    ci=np.tanh(z+np.array([-1,1])*stats.norm.ppf(.975)*se)
    print(r, 't=',t,'df=',dfree,'p=',p,'95% Fisher CI=',ci)


0.7 t= 2.191785018798023 df= 5 p= 0.07991669030889928 95% Fisher CI= [-0.11220696  0.95148936]
-0.12 t= -0.27028123880866767 df= 5 p= 0.7977452711162913 95% Fisher CI= [-0.80070117  0.69594891]
